# 04 — Fine-tune Chronos-2 with LoRA

**What this notebook does.** Takes Amazon's pretrained Chronos-2 forecasting model and
adapts it to our two financial series using LoRA, then pushes each adapted checkpoint to
the Hugging Face Hub.

## Before you run this, these must already exist

| Thing | Where | Made by |
|---|---|---|
| Processed series | `rohanjain2312/forecastbench-data` → `processed/spy_logrv.parquet`, `processed/dgs10.parquet` | build Step 15 |
| Model repo with its licence | `rohanjain2312/forecastbench-chronos` | build Step 15 |
| Colab Secrets | the 🔑 panel on the left | you, once |

If the dataset repo is empty, stop — run `scripts.fetch_data` and `data.hub.push_processed_series`
locally first. This notebook reads data, it does not build it.

## The one rule this notebook follows

**No modelling logic lives here.** Every cell below imports from `forecast_bench` and calls
a function. The loop over training blocks, the LoRA configuration, the early stopping, and
the checkpoint pushing all live in `forecast_bench/models/foundation/finetune.py`. If you
find yourself wanting to write a `for` loop over folds in a cell, it belongs in the package
instead — that is what stops this notebook and the repository from disagreeing.

## If Colab disconnects

Just run it again. Each block is pushed to the Hub as soon as it finishes, and startup
checks the Hub for what is already there. You lose one block, not the run.

## Step 1 — Confirm you actually have the GPU you asked for

Runtime → Change runtime type → **H100**. If this prints an error or shows a T4, the
fine-tuning will still work but will take considerably longer.

In [ ]:
!nvidia-smi

## Step 2 — Install the project

This pulls the exact code the benchmark runs locally, straight from GitHub, so the numbers
this notebook produces are generated by the same functions that produce the numbers in the
README. The `gpu` extras add `peft`, which does the LoRA maths.

Installed from a source tarball rather than `git+https://...`: pip's git codepath shells out to the container's git binary, which has been flaky inside Colab; fetching the tarball over plain HTTP avoids that dependency entirely.

`--force-reinstall --no-deps` matters as much as the tarball switch itself: without it, pip sees `forecast-bench` already installed (the version number never changes) and silently skips reinstalling, so reopening this notebook and clicking Run All can keep running *stale* code from an earlier session if Colab reconnects you to the same live runtime rather than a fresh one. `--no-deps` keeps this fast by leaving torch, darts and chronos-forecasting alone.


In [ ]:
%pip install -q --force-reinstall --no-deps "https://github.com/Rohanjain2312/forecast_bench/archive/refs/heads/main.tar.gz"
%pip install -q peft accelerate

# Colab preinstalls an old torchao (commonly 0.10.x). peft refuses to import at
# all unless torchao is at least 0.16.0, even though LoRA on attention
# projections never touches torchao's quantization features -- it is a hard
# import-time version check, not an actual dependency of what we run here.
%pip install -q -U "torchao>=0.16.0"

## Step 3 — Load your credentials from Colab Secrets

Click the 🔑 icon in the left sidebar and make sure `HF_TOKEN`, `FRED_API_KEY` and
`WANDB_API_KEY` are all there with **Notebook access** switched on.

Nothing is printed here. A token that appears in a notebook output is a token you have to
go and revoke.

In [ ]:
import os

from google.colab import userdata

from forecast_bench.config import get_config

for key in ["HF_TOKEN", "FRED_API_KEY", "WANDB_API_KEY"]:
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except Exception:
        print(f"{key}: not set (optional for W&B, required for the other two)")

os.environ.setdefault("HF_DATASET_REPO", "rohanjain2312/forecastbench-data")
os.environ.setdefault("HF_MODEL_REPO", "rohanjain2312/forecastbench-chronos")
os.environ.setdefault("HF_SPACE_REPO", "rohanjain2312/forecastbench-demo")

# get_config() is a process-wide singleton (functools.lru_cache) so that every
# caller in one run observes identical settings. That means if anything called
# it even once before this cell ran -- an earlier attempt, a re-run out of order --
# it is now permanently cached WITHOUT these secrets, and setting os.environ above
# would silently have no effect for the rest of this session. Clearing it here
# makes the notebook correct regardless of what was run before this cell.
get_config.cache_clear()

## Step 4 — Check what is already on the Hub

If you are re-running after a disconnect, this is where you will see the blocks that
already finished. They will be skipped automatically.

In [ ]:
from forecast_bench.config import setup_logging
from forecast_bench.models.foundation.finetune import existing_hub_revisions

setup_logging("INFO")
done = existing_hub_revisions()
print(f"{len(done)} checkpoints already on the Hub")
for tag in sorted(done):
    print(" ", tag)

## Step 5 — Fine-tune Chronos-2 on SPY log realized variance

One LoRA adapter per annual block, each trained only on data up to that block's forecast
origin. That restriction is the whole point: a model that trained on data from after the
origin it forecasts from would produce results that look excellent and mean nothing.

Each block is pushed to the Hub the moment it finishes.

The recipe — rank 8, alpha 16, dropout 0.05, early stopping with patience 3 on a validation
slice taken from the end of each block — is fixed in `finetune.py` and was written down
before any results existed. `PREREGISTRATION.md` §5 commits to not re-tuning it now.

In [ ]:
from forecast_bench.models.foundation.finetune import run_campaign

results = run_campaign(
    series="spy_logrv",
    model="chronos2",
    training_windows=("full",),
    device="cuda",
    num_steps=1000,
)

for r in results:
    status = "skipped (already on Hub)" if r.skipped else f"{r.trainable_parameters:,} trainable params ({r.trainable_fraction:.2%})"
    print(f"{r.tag}: {status}")

## Step 6 — Same for the 10-year Treasury yield

The contrast series. We expect this one to be hard — a near-unit-root series where the
random walk is difficult to beat — and reporting that honestly is the point of including it.

In [ ]:
results_dgs10 = run_campaign(
    series="dgs10",
    model="chronos2",
    training_windows=("full",),
    device="cuda",
    num_steps=1000,
)
print(f"{len(results_dgs10)} configurations finished")

## Step 7 — The sample-efficiency sweep

The claim worth testing: foundation models need far less adaptation data than from-scratch
models need training data. This trains the same model on 1 year, 3 years, 10 years, and the
full window, so the curve can be plotted against N-BEATS trained on the same four slices.

This is the longest cell in the notebook. It is also the one that produces the most legible
chart in the whole project — one axis is "how much data", the other is "how good".

In [ ]:
sweep = run_campaign(
    series="spy_logrv",
    model="chronos2",
    training_windows=("1y", "3y", "10y"),
    device="cuda",
    num_steps=1000,
)
print(f"{len(sweep)} sample-efficiency configurations finished")

## Step 8 — Chronos-Bolt, the older generation

Bolt takes the standard `transformers` + `peft` route rather than Chronos-2's own `fit`,
which is why the project fine-tunes both: two independent paths mean neither is a single
point of failure, and it turns a two-way comparison into a generational one.

**Known limitation, not a bug:** Bolt was only trained on quantiles 0.1–0.9, so it cannot
produce the study's 2.5% and 97.5% levels. Its 95% interval equals its 80% interval. This is
reported rather than worked around.

In [ ]:
bolt = run_campaign(
    series="spy_logrv",
    model="bolt",
    training_windows=("full",),
    device="cuda",
    num_steps=1000,
)
print(f"{len(bolt)} Chronos-Bolt configurations finished")

## Done

Every checkpoint is on `rohanjain2312/forecastbench-chronos`, each under its own revision
tag, so any result can be traced back to the exact weights that produced it.

**Tell Claude Code "done"**, then run notebook `05_colab_train_neural.ipynb`.